# MINI-PROJECT: DETEKSI TANDA TANGAN (SIGNATURE DETECTION)
**Mata Kuliah:** Pengolahan Citra Digital  
**Nama:** Salwa  
**NIM:** F1G124017  
**Kelas:** Kelas C

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

%matplotlib inline

In [ ]:
def crop_roi(img, y_min=0.50, y_max=0.90, x_min=0.55, x_max=0.95):
    """Memotong Region of Interest (ROI) tempat tanda tangan berada."""
    h, w = img.shape[:2]
    ymin, ymax = int(y_min * h), int(y_max * h)
    xmin, xmax = int(x_min * w), int(x_max * w)
    return img[ymin:ymax, xmin:xmax]

def process_signature_detection(img_bgr):
    """Pipeline pemrosesan citra digital untuk deteksi tanda tangan."""
    # 1. ROI Cropping
    roi_bgr = crop_roi(img_bgr)
    
    # 2. Grayscale Conversion
    gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
    
    # 3. Thresholding Comparison
    _, thresh_global = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 4. Morphological Operations (Opening & Closing)
    kernel_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    
    morph_open = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel_open)
    morph_close = cv2.morphologyEx(morph_open, cv2.MORPH_CLOSE, kernel_close)
    
    # 5. Feature Extraction (Ink Pixel Ratio)
    total_pixels = morph_close.size
    ink_pixels = cv2.countNonZero(morph_close)
    ink_ratio = (ink_pixels / total_pixels) * 100.0
    
    # 6. Decision Rule
    status = "SIGNATURE PRESENT" if ink_ratio >= 1.5 else "SIGNATURE ABSENT"
    
    return {
        'roi_bgr': roi_bgr,
        'gray': gray,
        'thresh_global': thresh_global,
        'thresh_otsu': thresh_otsu,
        'morph_close': morph_close,
        'ink_ratio': ink_ratio,
        'status': status
    }

In [ ]:
def load_dataset(data_dir):
    """Memuat citra dari folder data."""
    images = {}
    image_paths = glob.glob(os.path.join(data_dir, '*.*'))
    for path in image_paths:
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            filename = os.path.basename(path)
            img = cv2.imread(path)
            if img is not None:
                images[filename] = img
            else:
                print(f"Gagal memuat gambar: {filename}")
    return images

In [ ]:
# Load dataset from 'data' folder
data_folder = 'data'
dataset = load_dataset(data_folder)

print("=== EVALUASI DETEKSI TANDA TANGAN ===")
print("Nama : Salwa | NIM: F1G124017 | Kelas: C\n")

if not dataset:
    print(f"Peringatan: Tidak ada gambar ditemukan di folder '{data_folder}'")
else:
    for label, img in sorted(dataset.items()):
        res = process_signature_detection(img)
        print(f"Sampel: {label:<35} | Ink Ratio: {res['ink_ratio']:5.2f}% | Status: {res['status']}")

In [ ]:
# Visualisasi beberapa hasil dari dataset
if dataset:
    # Ambil hingga 3 gambar pertama untuk divisualisasikan
    sample_keys = list(sorted(dataset.keys()))[:3]
    
    for key in sample_keys:
        img = dataset[key]
        res = process_signature_detection(img)
        
        plt.figure(figsize=(15, 4))
        
        # Tampilkan Gambar Asli (RGB)
        plt.subplot(1, 4, 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f'Original\n{key[:15]}...')
        plt.axis('off')
        
        # Tampilkan ROI
        plt.subplot(1, 4, 2)
        plt.imshow(cv2.cvtColor(res['roi_bgr'], cv2.COLOR_BGR2RGB))
        plt.title('ROI')
        plt.axis('off')
        
        # Tampilkan Thresholding Otsu
        plt.subplot(1, 4, 3)
        plt.imshow(res['thresh_otsu'], cmap='gray')
        plt.title('Otsu Threshold')
        plt.axis('off')
        
        # Tampilkan Morphological Closing
        plt.subplot(1, 4, 4)
        plt.imshow(res['morph_close'], cmap='gray')
        plt.title(f"Morph Close\nInk Ratio: {res['ink_ratio']:.2f}%")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()